# Bouncy Castle FAQ – RAG Pipeline Exploration

This notebook walks through the completed ingest–search pipeline step by step.
It demonstrates every function from tasks 1–5 without running the test suite.

## 1. Setup

Import all public functions from the project's `src` package and create a temporary working directory for indexes and the DuckDB database.

In [1]:
import json
import pathlib
import pickle
import shutil

import duckdb
import faiss

from src.faqs import load_faqs
from src.pipeline import run_pipeline
from src.ingest import build_indexes
from src.search import search

tmp_dir = pathlib.Path(".tmp")
tmp_dir.mkdir(exist_ok=True)
print(f"Working directory: {tmp_dir.resolve()}")

/Users/henrik/Documents/dev/python/llm-zoomcamp/llm-zoomcamp-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working directory: /Users/henrik/Documents/dev/python/llm-zoomcamp/llm-zoomcamp-rag/.tmp


## 2. Load FAQ Data

Use `load_faqs()` to read the CSV, verify row count (42), inspect a few sample rows, and check for missing values.

In [2]:
faqs = load_faqs()
print(f"Rows: {len(faqs)}")
print(f"Columns: {', '.join(faqs[0].keys())}")
print()
print("--- 3 Sample Rows ---")
for row in faqs[:3]:
    print(f"  [{row['Category']}] {row['Question']}")
    print(f"  Answer: {row['Answer'][:80]}...")
    print()

missing = {k: sum(1 for row in faqs if not row.get(k)) for k in faqs[0]}
print("--- Missing Values per Column ---")
for col, count in missing.items():
    print(f"  {col}: {count} missing")

Rows: 41
Columns: Category, Question, Answer

--- 3 Sample Rows ---
  [Booking & Reservations] How far in advance should I book?
  Answer: It's recommended to book as early as possible. Many companies accept bookings up...

  [Booking & Reservations] Is a deposit required to reserve a bouncy castle?
  Answer: Yes. Most companies require a non-refundable deposit (typically €50 or 10-50% of...

  [Booking & Reservations] What payment methods are accepted?
  Answer: Most rental companies accept credit cards. bank transfers. cash. and PayPal. Som...

--- Missing Values per Column ---
  Category: 0 missing
  Question: 0 missing
  Answer: 0 missing


## 3. dlt Pipeline

Run the dlt pipeline into a local DuckDB file inside `.tmp/`. Query the loaded table and confirm idempotency by re-running the pipeline.

In [3]:
db_path = tmp_dir / "test_faq.duckdb"
pipeline, info = run_pipeline(
    destination="duckdb",
    dataset_name="faq",
    pipeline_name="test_faq_pipeline",
    db_path=db_path,
)
print("First run:")
print(info)

con = duckdb.connect(str(db_path))
count = con.sql('SELECT COUNT(*) FROM "faq"."faq_resource"').fetchone()[0]
print(f"Rows loaded: {count}")
print()

categories = con.sql('SELECT DISTINCT Category FROM "faq"."faq_resource" ORDER BY Category').fetchall()
print("Categories in the dataset:")
for (cat,) in categories:
    print(f"  - {cat}")
con.close()

First run:
Pipeline test_faq_pipeline load step completed in 0.05 seconds
1 load package(s) were loaded to destination duckdb and into dataset faq
The duckdb destination used duckdb:////Users/henrik/Documents/dev/python/llm-zoomcamp/llm-zoomcamp-rag/.tmp/test_faq.duckdb location to store data
Load package 1785355973.546321 is LOADED and contains no failed jobs
Rows loaded: 41

Categories in the dataset:
  - Booking & Reservations
  - Cancellation & Weather Policies
  - Cleaning & Damage
  - Delivery Setup & Pickup
  - General Questions
  - Insurance & Liability
  - Location-Specific Questions
  - Pricing & Additional Fees
  - Safety Rules & Supervision
  - Setup Requirements


In [4]:
_pipeline2, _info2 = run_pipeline(
    destination="duckdb",
    dataset_name="faq",
    pipeline_name="test_faq_pipeline",
    db_path=db_path,
)
con2 = duckdb.connect(str(db_path))
count2 = con2.sql('SELECT COUNT(*) FROM "faq"."faq_resource"').fetchone()[0]
con2.close()

print(f"Rows after re-run: {count2}")
print(f"Idempotent (row count unchanged): {count2 == count}")

Rows after re-run: 41
Idempotent (row count unchanged): True


## 4. Build Indexes

Build BM25 and FAISS indexes from the FAQ data, saving them into `.tmp/`. Then inspect each index to confirm structure.

In [5]:
bm25_path = tmp_dir / "bm25_index.pkl"
faiss_path = tmp_dir / "faiss_index.bin"
docs_path = tmp_dir / "ingest_docs.json"

result = build_indexes(
    faqs=faqs,
    bm25_path=bm25_path,
    faiss_path=faiss_path,
    docs_path=docs_path,
    force=True,
)
print("Index files created:")
for k, v in result.items():
    print(f"  {k}: {v}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19567.59it/s]


Index files created:
  bm25_path: .tmp/bm25_index.pkl
  faiss_path: .tmp/faiss_index.bin
  docs_path: .tmp/ingest_docs.json


In [6]:
with open(bm25_path, "rb") as f:
    bm25 = pickle.load(f)
print(f"BM25 corpus size: {bm25.corpus_size} documents")
print(f"BM25 vocabulary size: {len(bm25.idf)} tokens")
print()

index = faiss.read_index(str(faiss_path))
print(f"FAISS index ntotal: {index.ntotal} vectors")
print(f"FAISS index dimension: {index.d}")
print()

with open(docs_path, encoding="utf-8") as f:
    docs_data = json.load(f)
print(f"Docs JSON entries: {len(docs_data)}")
print(f"First doc keys: {list(docs_data[0].keys())}")
print(f"Sample doc: {json.dumps(docs_data[0], ensure_ascii=False)}")

BM25 corpus size: 41 documents
BM25 vocabulary size: 516 tokens

FAISS index ntotal: 41 vectors
FAISS index dimension: 384

Docs JSON entries: 41
First doc keys: ['text', 'category', 'question', 'answer']
Sample doc: {"text": "Booking & Reservations: How far in advance should I book? It's recommended to book as early as possible. Many companies accept bookings up to a year in advance with a deposit. Popular dates fill up quickly.", "category": "Booking & Reservations", "question": "How far in advance should I book?", "answer": "It's recommended to book as early as possible. Many companies accept bookings up to a year in advance with a deposit. Popular dates fill up quickly."}


## 5. Hybrid Search

Search using BM25 + FAISS with Reciprocal Rank Fusion (RRF). Try different queries, compare `k=1` vs `k=5`, and test the empty-query edge case.

In [7]:
def show_results(query, results):
    print(f"Query: {query!r}")
    print(f"Results: {len(results)}")
    for i, r in enumerate(results, 1):
        print(f"  {i}. [{r['category']}] {r['question']}")
        print(f"     Score: {r['score']}  |  Answer: {r['answer'][:60]}...")
    print()

In [8]:
results_booking = search("booking", k=5, bm25_path=bm25_path, faiss_path=faiss_path, docs_path=docs_path)
show_results("booking", results_booking)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8366.51it/s]


Query: 'booking'
Results: 5
  1. [Booking & Reservations] Do you have a referral program?
     Score: 0.0333  |  Answer: Some companies offer referral credits (e.g. €25 per referral...
  2. [Booking & Reservations] What payment methods are accepted?
     Score: 0.0328  |  Answer: Most rental companies accept credit cards. bank transfers. c...
  3. [Booking & Reservations] Do you offer discounts for multiple-day rentals?
     Score: 0.032  |  Answer: Yes. many companies offer discounted rates for multi-day ren...
  4. [Booking & Reservations] How far in advance should I book?
     Score: 0.032  |  Answer: It's recommended to book as early as possible. Many companie...
  5. [Setup Requirements] What if my setup location is inaccessible or unsafe?
     Score: 0.0156  |  Answer: If the delivery team cannot safely set up due to inaccessibl...



In [9]:
results_cost = search("cost", k=5, bm25_path=bm25_path, faiss_path=faiss_path, docs_path=docs_path)
show_results("cost", results_cost)

Query: 'cost'
Results: 5
  1. [Pricing & Additional Fees] How much does it cost to rent a bouncy castle?
     Score: 0.0325  |  Answer: Prices vary by size. type. and location: Small basic bounce ...
  2. [Setup Requirements] What type of surface can the bouncy castle be set up on?
     Score: 0.0167  |  Answer: Most companies can set up on grass. concrete. asphalt. or in...
  3. [Pricing & Additional Fees] Are there any hidden fees I should know about?
     Score: 0.0167  |  Answer: Common additional charges: delivery fees for out-of-area loc...
  4. [Pricing & Additional Fees] What is included in the rental price?
     Score: 0.0164  |  Answer: Most rental prices include the inflatable unit. delivery. se...
  5. [General Questions] Can I use a water slide if it's not hot outside?
     Score: 0.0161  |  Answer: Water slides can be used as dry slides if the weather is coo...



In [10]:
results_safety = search("safety", k=5, bm25_path=bm25_path, faiss_path=faiss_path, docs_path=docs_path)
show_results("safety", results_safety)

Query: 'safety'
Results: 5
  1. [Safety Rules & Supervision] Is adult supervision required?
     Score: 0.0328  |  Answer: Yes. adult supervision (age 18+) is required at all times wh...
  2. [Safety Rules & Supervision] Why is silly string prohibited?
     Score: 0.0315  |  Answer: Silly string is highly flammable and can damage the inflatab...
  3. [Insurance & Liability] Am I responsible for injuries or damage?
     Score: 0.0167  |  Answer: The renter assumes responsibility for the safety of all user...
  4. [Safety Rules & Supervision] Do children need to wear socks?
     Score: 0.0164  |  Answer: Yes. socks are typically required to protect children's feet...
  5. [Safety Rules & Supervision] What items are prohibited in the bouncy castle?
     Score: 0.0164  |  Answer: Face paints. glitter. silly string. party poppers. confetti....



In [11]:
print("--- k=1 vs k=5 ---")
r1 = search("deposit", k=1, bm25_path=bm25_path, faiss_path=faiss_path, docs_path=docs_path)
print(f"k=1 returned {len(r1)} result{'s' if len(r1)!=1 else ''}: {r1[0]['question'] if r1 else '(none)'}")
print()
r5 = search("deposit", k=5, bm25_path=bm25_path, faiss_path=faiss_path, docs_path=docs_path)
print(f"k=5 returned {len(r5)} results:")
for r in r5:
    print(f"  - {r['question']}  (score: {r['score']})")

--- k=1 vs k=5 ---
k=1 returned 1 result: Is a deposit required to reserve a bouncy castle?

k=5 returned 5 results:
  - Is a deposit required to reserve a bouncy castle?  (score: 0.0333)
  - What is the cancellation policy?  (score: 0.0325)
  - How far in advance should I book?  (score: 0.032)
  - What payment methods are accepted?  (score: 0.0164)
  - What happens if it rains or there's bad weather?  (score: 0.0159)


In [12]:
print("--- Empty Query Edge Case ---")
empty_results = search("", k=5, bm25_path=bm25_path, faiss_path=faiss_path, docs_path=docs_path)
print(f"Empty query returned {len(empty_results)} result{'s' if len(empty_results)!=1 else ''}")
if empty_results:
    for r in empty_results:
        print(f"  - {r['question']}  (score: {r['score']})")
else:
    print("  (no results — empty query handled gracefully)")

--- Empty Query Edge Case ---
Empty query returned 5 results
  - Can I use a water slide if it's not hot outside?  (score: 0.0167)
  - What is included in the rental price?  (score: 0.0167)
  - What are the capacity limits?  (score: 0.0164)
  - Who sets up the bouncy castle?  (score: 0.0164)
  - Can I keep the bouncy castle overnight?  (score: 0.0161)


## 6. Cleanup

Remove the `.tmp/` directory and all its contents (indexes, DuckDB database).

In [13]:
shutil.rmtree(tmp_dir, ignore_errors=True)
print(f"Removed {tmp_dir.resolve()}")

Removed /Users/henrik/Documents/dev/python/llm-zoomcamp/llm-zoomcamp-rag/.tmp
